# ⚽ Agente RAG: Regras do Futebol

Assistente que responde perguntas sobre as **Regras do Jogo** de futebol (IFAB, edição 2023/24, em português) usando **RAG (Retrieval-Augmented Generation)**.

Em vez de depender só do que o modelo de linguagem "lembra", o agente busca os trechos relevantes do documento oficial e responde **com base neles**, indicando a página de onde tirou a informação. Quando a pergunta não é coberta pelas regras, ele diz isso em vez de inventar.

**Fluxo:**

```
PDF das Regras → divisão em trechos → embeddings → banco vetorial (Chroma)
                                                          │
Pergunta → busca dos trechos mais parecidos ──────────────┘
         → prompt com o contexto → LLM → resposta com a página de origem
```

**Tecnologias:** LangChain (LCEL) · OpenAI (`gpt-4o-mini` e `text-embedding-3-small`) · Chroma · PyPDF

## 1. Instalação

In [ ]:
%pip install -q "langchain-core>=1.0,<2" "langchain-openai>=1.0,<2" "langchain-chroma>=1.0,<2" \
    "langchain-community>=0.4,<1" "langchain-text-splitters>=1.0,<2" "chromadb>=1.0,<2" "pypdf>=6,<7"

## 2. Chave da API da OpenAI

No Colab, a chave é lida dos **Secrets** (ícone 🔑 na barra lateral), com o nome `OPENAI_API_KEY`. Fora do Colab, ela é lida da variável de ambiente ou solicitada na hora. A chave nunca fica escrita no código.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from getpass import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

## 3. Configuração

In [ ]:
from pathlib import Path

CAMINHO_PDF = Path("regras_futebol.pdf")
URL_PDF = "https://downloads.theifab.com/downloads/laws-of-the-game-2023-24-portuguese?l=en"

MODELO_EMBEDDINGS = "text-embedding-3-small"
MODELO_LLM = "gpt-4o-mini"

TAMANHO_TRECHO = 500      # caracteres por trecho
SOBREPOSICAO = 100        # caracteres repetidos entre trechos vizinhos
TRECHOS_POR_PERGUNTA = 3  # quantos trechos são enviados ao LLM

## 4. Carregamento do PDF

Se o arquivo ainda não estiver na pasta, ele é baixado do site oficial da IFAB. Caso o download falhe, faça o upload manual de `regras_futebol.pdf` pelo painel de arquivos do Colab.

In [ ]:
import urllib.request

from langchain_community.document_loaders import PyPDFLoader


def obter_pdf(caminho: Path, url: str) -> Path:
    """Baixa o PDF se ele ainda não existir e confere se o arquivo é mesmo um PDF."""
    if not caminho.exists():
        print("Baixando o PDF das Regras do Jogo...")
        requisicao = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(requisicao) as resposta:
            caminho.write_bytes(resposta.read())

    with caminho.open("rb") as arquivo:
        if arquivo.read(4) != b"%PDF":
            caminho.unlink()
            raise ValueError(
                f"O arquivo obtido em {url} não é um PDF. "
                f"Faça o upload manual de {caminho.name} e rode esta célula de novo."
            )
    return caminho


paginas = PyPDFLoader(str(obter_pdf(CAMINHO_PDF, URL_PDF))).load()
print(f"{len(paginas)} páginas carregadas")

## 5. Divisão em trechos

O documento é dividido em trechos menores para que a busca encontre a parte exata que responde a cada pergunta. A sobreposição evita que uma frase importante fique cortada entre dois trechos.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def pagina(documento) -> str:
    """Número da página como aparece no PDF."""
    return documento.metadata.get("page_label") or str(documento.metadata.get("page", 0) + 1)


divisor = RecursiveCharacterTextSplitter(chunk_size=TAMANHO_TRECHO, chunk_overlap=SOBREPOSICAO)
trechos = divisor.split_documents(paginas)

print(f"{len(trechos)} trechos gerados\n")
exemplo = trechos[len(trechos) // 2]
print(f"Exemplo (página {pagina(exemplo)}):\n{exemplo.page_content}")

## 6. Embeddings e banco vetorial

Cada trecho é convertido em um vetor numérico que representa seu significado e guardado no Chroma. A coleção é limpa antes de receber os trechos, para que rodar a célula de novo não duplique os documentos.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=MODELO_EMBEDDINGS)

banco = Chroma(collection_name="regras_futebol", embedding_function=embeddings)
banco.reset_collection()
banco.add_documents(trechos)

buscador = banco.as_retriever(search_kwargs={"k": TRECHOS_POR_PERGUNTA})
print(f"{len(trechos)} trechos indexados")

## 7. Cadeia RAG

A cadeia busca os trechos mais relevantes, monta o prompt com eles e envia ao modelo. O prompt instrui o modelo a responder **somente** com base no contexto, citar a página e admitir quando a informação não está nas regras.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Você é um assistente especializado nas Regras do Jogo de futebol da IFAB (edição 2023/24). "
     "Responda em português, de forma clara e objetiva, usando APENAS os trechos do contexto abaixo. "
     "Indique a página em que a resposta se baseia. "
     "Se o contexto não trouxer a informação, diga que ela não consta nas Regras do Jogo, sem inventar.\n\n"
     "Contexto:\n{contexto}"),
    ("human", "{pergunta}"),
])

llm = ChatOpenAI(model=MODELO_LLM, temperature=0)


def formatar_contexto(entrada: dict) -> dict:
    contexto = "\n\n".join(f"[Página {pagina(t)}]\n{t.page_content}" for t in entrada["trechos"])
    return {"contexto": contexto, "pergunta": entrada["pergunta"]}


gerar_resposta = formatar_contexto | prompt | llm | StrOutputParser()

cadeia_rag = (
    RunnableParallel(trechos=buscador, pergunta=RunnablePassthrough())
    .assign(resposta=gerar_resposta)
)

## 8. Testes

In [ ]:
def perguntar(pergunta: str, mostrar_trechos: bool = True) -> str:
    """Faz uma pergunta ao agente e mostra a resposta e os trechos usados como contexto."""
    resultado = cadeia_rag.invoke(pergunta)

    print(f"❓ {pergunta}\n")
    print(f"💬 {resultado['resposta']}\n")

    if mostrar_trechos:
        for i, trecho in enumerate(resultado["trechos"], start=1):
            print(f"--- Trecho {i} (página {pagina(trecho)}) ---")
            print(trecho.page_content, "\n")

    return resultado["resposta"]

In [ ]:
perguntar("Um jogador pode usar a mão para marcar um gol?")

In [ ]:
perguntar("O que caracteriza a posição de impedimento?")

In [ ]:
perguntar("Qual é o número mínimo de jogadores para uma partida continuar?")

**Pergunta fora do escopo:** o agente deve reconhecer que a informação não está nas Regras do Jogo, em vez de responder com o conhecimento geral do modelo.

In [ ]:
perguntar("Quem ganhou a Copa do Mundo de 2022?", mostrar_trechos=False)